In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

passenger_ids = test["PassengerId"]

In [3]:
train

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [4]:
test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [5]:
train.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
def data_prep(df):
    df['Age'] = df['Age'].fillna(df.groupby('Sex')['Age'].transform('mean'))

    df = df.drop(columns=['Cabin', 'Name','PassengerId', 'Ticket'])
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare'] = np.log1p(df['Fare'])
    df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)
    
    X = df.drop(columns=['Survived'], errors='ignore')

    return X

In [8]:
x_train, y_train = data_prep(train), train['Survived']
x_test = data_prep(test)

In [9]:
x_test[x_test.isnull().any(axis=1)]

,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
152,3,60.5,0,0,NaN,1,0,1


In [10]:
# Заполнение пропуска медианой (рекомендуется)
x_test['Fare'] = x_test['Fare'].fillna(x_train['Fare'].median())

x_test[x_test.isnull().any(axis=1)]


,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S


In [11]:
# 1. Создаем и обучаем масштабировщик (только на тренировочных данных!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

# 2. Инициализируем модель
model = LogisticRegression()

# 3. Обучаем модель НА ОТМАСШТАБИРОВАННЫХ данных
model.fit(X_train_scaled, y_train)

# 4. Делаем предсказания тоже НА ОТМАСШТАБИРОВАННЫХ данных
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# Получаем вероятности для тест и трейн выборок (индекс [:, 1] берет класс "1" — выжил)
y_proba_test = model.predict_proba(X_test_scaled)[:, 1]
y_proba_train = model.predict_proba(X_train_scaled)[:, 1]

# Считаем корректный ROC-AUC
print("ROC-AUC на трейне:", round(roc_auc_score(y_train, y_proba_train),4))


ROC-AUC на трейне: 0.8615


In [13]:
pred = model.predict(X_test_scaled)

submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": pred
})

In [14]:
submission.to_csv("submission.csv", index=False)

catboost trying:

In [19]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    verbose=0,
    random_seed=42
)

model.fit(x_train, y_train)

from sklearn.metrics import accuracy_score, roc_auc_score

pred = model.predict(x_test)

submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": pred.astype(int)
})

submission.to_csv("submission_catboost.csv", index=False)